# DataMate · toàn bộ quy trình (01 → 05)

Một notebook chạy lần lượt cả năm bước của baseline Track 2 (Zalo AI Challenge 2026, DataMate), ghép tự động từ các notebook
`notebooks/01_eda` … `05_train_lora` bằng `scripts/build_full_notebook.py`:

| Phần | Việc | Phần cứng |
|---|---|---|
| 01 · EDA | đọc dữ liệu BTC: shop, tệp, loại đáp án, dạng câu hỏi, rulebook | CPU |
| 02 · Suy luận | pipeline `datamate` trên một split, ghi `submission.jsonl` | GPU bf16 từ 24 GB |
| 03 · Đánh giá | điểm một lượt chạy train, nhóm lỗi, ngưỡng bỏ phiếu; so các bản nộp public | CPU |
| 04 · Dữ liệu train LoRA | dòng SFT từ lượt chạy train có nhãn, ghi chú trước SQL | CPU, GPU cho bước ghi chú |
| 05 · Train LoRA | QLoRA trên Qwen3.5-9B bằng Unsloth | GPU bf16 từ 24 GB |

Mỗi phần có ô tham số riêng ngay đầu phần (giá trị mặc định là cấu hình của bản đã nộp). Chạy từ trên xuống; phần nào không cần có thể bỏ qua.
Trên Kaggle, phần 05 cài stack train khác với vLLM của 02 / 04: chạy 05 trong một phiên riêng (khởi động lại, chạy ô setup rồi tới phần 05).

In [ ]:
# Setup. Local checkout: nothing to install. Kaggle: Internet on, GPU accelerator for inference / training, the competition data
# attached as a private dataset, and the code either attached as a dataset (a copy of the checkout) or cloned with a team member's
# GitHub token in Add-ons -> Secrets as GITHUB_TOKEN (the repo is private). Colab: Google Drive holding MyDrive/datamate/code and
# MyDrive/datamate/data (or a GITHUB_TOKEN secret for the code), a GPU runtime for inference / training.
import os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "github.com/hugebenevolence/zalo-ai-challenge-datamate-26.git"


def clone(root, token):
    subprocess.run(["git", "clone", "--depth", "1", f"https://x-access-token:{token}@{REPO_URL}", str(root)], check=True)
    if subprocess.run(["git", "lfs", "version"], capture_output=True).returncode:
        subprocess.run("apt-get -qq update && apt-get -qq install -y git-lfs", shell=True, check=True)
    subprocess.run(["git", "lfs", "install", "--local"], cwd=root, check=True)
    subprocess.run(["git", "lfs", "pull"], cwd=root, check=True)                                # the LoRA adapter
    subprocess.run(["git", "remote", "set-url", "origin", f"https://{REPO_URL}"], cwd=root, check=True)  # token out of saved files


if Path("/kaggle/input").exists():
    root = Path("/tmp/zalo-ai-challenge-datamate-26")
    attached = sorted(p.parent.parent for p in Path("/kaggle/input").glob("**/datamate/notebook.py"))
    if not root.exists() and attached:
        shutil.copytree(attached[0], root)                         # code attached as a dataset: no token needed
    if not root.exists():
        from kaggle_secrets import UserSecretsClient
        clone(root, UserSecretsClient().get_secret("GITHUB_TOKEN"))
elif "COLAB_RELEASE_TAG" in os.environ:
    from google.colab import drive
    drive.mount("/content/drive")
    root, code = Path("/content/zalo-ai-challenge-datamate-26"), Path("/content/drive/MyDrive/datamate/code")
    if not root.exists() and (code / "datamate/notebook.py").exists():
        shutil.copytree(code, root)                                # code uploaded to Drive: no token needed
    if not root.exists():
        from google.colab import userdata                          # key icon -> secret GITHUB_TOKEN with notebook access
        clone(root, userdata.get("GITHUB_TOKEN"))
else:
    root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(root)
sys.path.insert(0, str(root))
from datamate.notebook import install, setup
install("base")            # each part below installs what it needs (Kaggle only)
env = setup()

In [ ]:
import json
from collections import Counter

import matplotlib.pyplot as plt
import pandas as pd

read = lambda p: [json.loads(l) for l in open(p, encoding="utf-8-sig") if l.strip()]
pd.set_option("display.max_colwidth", 80)

## 01 · Khám phá dữ liệu (EDA)

Đọc dữ liệu BTC: số shop và câu hỏi, tệp của từng shop, loại đáp án, dạng câu hỏi, khác biệt rulebook, một shop sau khi nạp.

In [ ]:
train_q, public_q = read(env.data / "train/questions.jsonl"), read(env.data / "public_test/questions.jsonl")
train_a = {a["id"]: a for a in read(env.data / "train/answers.jsonl")}
splits = {"train": train_q, "public": public_q}
def per_shop(qs):
    c = list(Counter(q["business_id"] for q in qs).values())
    return f"{min(c)} / {pd.Series(c).median():.0f} / {max(c)}"
pd.DataFrame({name: {"questions": len(qs), "shops": len({q["business_id"] for q in qs}), "questions per shop (min / median / max)": per_shop(qs),
                     "as_of_date range": f'{min(q["as_of_date"] for q in qs)} .. {max(q["as_of_date"] for q in qs)}'} for name, qs in splits.items()})

### Tệp dữ liệu của shop

Mỗi shop là một thư mục gồm README.md và các bảng (csv / xlsx / sqlite). Số bảng và dung lượng khác nhau rất nhiều.

In [ ]:
rows = []
for split, folder in (("train", "train"), ("public", "public_test")):
    for shop in sorted((env.data / folder / "data").iterdir()):
        files = [f for f in shop.rglob("*") if f.is_file() and f.name != "README.md"]
        rows.append({"split": split, "shop": shop.name, "files": len(files), "MB": round(sum(f.stat().st_size for f in files) / 2**20, 1),
                     **Counter(f.suffix.lower().lstrip(".") for f in files)})
files = pd.DataFrame(rows).fillna(0)
display(files.groupby("split")[[c for c in files.columns if c not in ("split", "shop")]].sum())
files.groupby("split")[["files", "MB"]].describe().round(1)

### Loại đáp án (train)

Phân bố type của đáp án chuẩn. `alias` là các cách viết khác được chấp nhận (mã hàng, cách viết khác trong dữ liệu).

In [ ]:
types = Counter(a["type"] for a in train_a.values())
pd.Series(types).sort_values().plot.barh(figsize=(6, 3), title="train answer types")
print({t: f"{n / len(train_a):.1%}" for t, n in types.most_common()})
print("answers with aliases:", sum(bool(a.get("alias")) for a in train_a.values()))

### Dạng câu hỏi

Tỷ lệ câu kích hoạt từng gợi ý trong `datamate/hints.py` và `datamate/prompts.py`. Dạng câu có trong cả train và public thì kiểm chứng
được bằng đáp án train. Dạng chỉ có ở public (tiền viết tắt, từ địa phương) thì không.

In [ ]:
import re
from datamate import hints as H
from datamate.ingest import norm
from datamate.prompts import SLANG, money_gloss
shapes = {
    "period named": lambda q: bool(H.period_hints(q["question"], q["as_of_date"])),
    "debt": lambda q: bool(H.DEBT_WORDS.search(norm(q["question"]))),
    "days by daily total": lambda q: bool(H.COUNT_DAYS.search(norm(q["question"])) and H.THRESHOLD.search(norm(q["question"]) + q["question"])),
    "profit": lambda q: bool(H.PROFIT.search(norm(q["question"]))),
    "delivery trips": lambda q: bool(H.TRIPS.search(norm(q["question"]))),
    "lots / expiry": lambda q: bool(H.LOT.search(norm(q["question"]))),
    "money shorthand": lambda q: bool(money_gloss(q["question"])),
    "chat words": lambda q: bool(set(re.findall(r"\w+", q["question"].lower())) & set(SLANG)),
}
share = pd.DataFrame({name: {k: sum(f(q) for q in qs) / len(qs) for k, f in shapes.items()} for name, qs in splits.items()})
share.plot.barh(figsize=(7, 4), title="share of questions")
share.style.format("{:.1%}")

In [ ]:
pd.DataFrame({name: pd.Series([len(q["question"]) for q in qs]).describe().round(0) for name, qs in splits.items()}).T

### Rulebook

Train và public có rulebook riêng. Public thêm một số luật (khách trung thành, giờ cao điểm, nhà cung cấp chính…).

In [ ]:
rb = {s: (env.data / f / "rulebook.md").read_text(encoding="utf-8") for s, f in (("train", "train"), ("public", "public_test"))}
rules = {s: {l.split("|")[1].strip() for l in t.splitlines() if l.startswith("| ") and not l.startswith("| Luật")} for s, t in rb.items()}
print(len(rules["train"]), "rules in train,", len(rules["public"]), "in public")
print("only in public:", sorted(rules["public"] - rules["train"]))
print("only in train:", sorted(rules["train"] - rules["public"]))

### Một shop sau khi nạp

`load_business` đọc mọi bảng vào DuckDB, `schema_text` là phần SCHEMA mà model nhìn thấy.

In [ ]:
from datamate.ingest import load_business, schema_text
db, meta = load_business(env.data / "train", "biz_101", canon=True)
display(pd.DataFrame([{"table": t["name"], "rows": t["rows"], "columns": len(t["columns"])} for t in meta["tables"]]))
print(schema_text(meta)[:3000])

## 02 · Suy luận (inference)

Chạy pipeline `datamate` trên một split và ghi `submission.jsonl`: nạp shop → view và lớp công nợ → 8 mẫu SQL thường + 4 mẫu thinking cho câu
khó → chạy, sửa lỗi, bỏ phiếu. Cờ lấy từ `scripts/flags.sh` (bản đã nộp, public 0.683) và tự chỉnh theo GPU.

**Phần cứng:** một GPU bf16 (Ampere trở lên) từ 24 GB cho mỗi tiến trình. Nhiều GPU thì mỗi GPU một phần shop. GPU Ada / Hopper / Blackwell
chạy fp8 như bản đã nộp, Ampere (A10, A100…) chạy bf16 với context ngắn hơn. T4 / P100 không chạy được model này.

**Thời gian:** model nạp vài phút, `LIMIT = 10` để thử, cả 1000 câu public khoảng 3,5 giờ trên một GPU 24 GB.

**Dùng cho train:** `SPLIT = "train"`, `REUSE_LAYERS = False`, `SAVE_PROMPTS = True` cho ra lượt chạy có nhãn mà `04_training_data` cần.

In [ ]:
install("llm")

In [ ]:
SPLIT = "public_test"        # public_test | train | private_test (a folder of the data dir)
LIMIT = 10                   # first N questions (None = all)
IDS = None                   # a file of question ids, e.g. "splits/rft_train24.txt" or "splits/holdout600.txt"
REUSE_LAYERS = True          # public only: views and debt layers of the 0.683 run (configs/public_v9). False = draft them (as on private)
LORA = "sql_rft_v3"          # an adapter under adapters/, or the path of one trained in 05_train_lora
SAVE_PROMPTS = True          # keep each question's prompt in the trace (needed to build SFT rows from a train run)
OUT_NAME = "nb_" + SPLIT

In [ ]:
model, lora = env.base_model(), env.adapter(LORA)
flags, engine = env.inference_flags()
out = env.runs / OUT_NAME
n = len(env.gpus)
layers = ["--views-from", "configs/public_v9"] if REUSE_LAYERS and SPLIT == "public_test" else []
pick = (["--limit", str(LIMIT)] if LIMIT else []) + (["--ids", IDS] if IDS else []) + (["--save-prompts"] if SAVE_PROMPTS else [])
make_cmd = lambda i: [env.python, "-m", "datamate.predict", "--data", env.data / SPLIT, "--output", out / f"shard{i}", "--shard", f"{i}/{n}",
                      "--model", model, "--lora", lora, "--fewshot", "assets/fewshot_bank.jsonl", *flags, "--engine", engine, *layers, *pick]
print(f"{n} GPU process(es) -> {out}\nflags: {' '.join(flags)}")

In [ ]:
env.run_shards(make_cmd, n, out)

In [ ]:
from datamate.submission import check, merge_shards
merge_shards(out, env.data / SPLIT / "questions.jsonl")
errors = check(out / "submission.jsonl", env.data / SPLIT / "questions.jsonl")
print("format OK" if not errors else errors[:5])

### Kết quả

Số câu đã trả lời theo shop, và vài câu mẫu (đáp án, tỷ lệ phiếu đồng ý). Với `SPLIT = "train"` có thêm điểm theo đáp án train.

In [ ]:
trace = [r for p in sorted(out.glob("shard*/trace.jsonl")) for r in read(p)]
df = pd.DataFrame([{"shop": r["business_id"], "answered": r["final"]["type"] != "abstain", "conf": r["conf"]} for r in trace])
display(df.groupby("shop").agg(questions=("answered", "size"), answered=("answered", "sum"), mean_conf=("conf", "mean")).round(2))
pd.DataFrame([{"question": r["question"][:90], "final": json.dumps(r["final"], ensure_ascii=False)[:80], "conf": round(r["conf"], 2)} for r in trace[:5]])

In [ ]:
if (env.data / SPLIT / "answers.jsonl").exists():
    from datamate import scoring
    gold = {a["id"]: a for a in read(env.data / SPLIT / "answers.jsonl")}
    report, _ = scoring.evaluate({r["id"]: gold[r["id"]] for r in trace}, {r["id"]: r["final"] for r in trace},
                                 {q["id"]: q for q in read(env.data / SPLIT / "questions.jsonl")})
    print({k: report[k] for k in ("N", "correct", "wrong", "abstain", "PA", "precision")})

## 03 · Đánh giá

1. Một lượt chạy trên train (có đáp án): điểm, theo loại / shop / dạng câu, nhóm lỗi, ngưỡng bỏ phiếu và oracle@K.
2. Các bản nộp public: so với nhau, không cần nhãn (điểm public chỉ có ở mức tổng, không bao giờ làm nhãn cho từng câu).

In [ ]:
RUN = "runs/sql_v8_pool"     # a train run (its trace.jsonl, or shard*/trace.jsonl), e.g. runs/nb_train from 02_inference
SUBS = {"v6 (0.63)": "runs/sub_v6", "v8 (0.639)": "runs/sub_v8", "v9 (0.683)": "runs/sub_v9"}   # public submissions to compare

In [ ]:
from datamate import scoring, voting
from datamate.hints import DEBT_WORDS
from datamate.ingest import norm
run = env.root / RUN if not str(RUN).startswith("/") else Path(RUN)
gold = {a["id"]: a for a in read(env.data / "train/answers.jsonl")}
trace = {r["id"]: r for p in ([run / "trace.jsonl"] if (run / "trace.jsonl").exists() else sorted(run.glob("shard*/trace.jsonl"))) for r in read(p)}
print(len(trace), "traced questions")

### 1. Điểm của lượt chạy train

`scoring.evaluate` cho PA, độ chính xác khi trả lời, khoảng tin cậy bootstrap theo shop và phân rã theo loại đáp án.

In [ ]:
questions = {q["id"]: q for q in read(env.data / "train/questions.jsonl")}
report, rows = scoring.evaluate({i: gold[i] for i in trace}, {i: r["final"] for i, r in trace.items()}, questions)
print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in report.items() if k in ("N", "correct", "wrong", "abstain", "PA", "precision", "coverage")})
print("PA 95% CI (shop bootstrap):", [round(x, 3) for x in report["PA_cluster_bootstrap_95"]])
pd.DataFrame(report["breakdowns"]["gold_type"]).T[["N", "correct", "wrong", "abstain", "PA"]].sort_values("N", ascending=False)

In [ ]:
by_shop = pd.DataFrame(report["breakdowns"]["business_id"]).T[["N", "correct", "wrong", "abstain", "PA"]].sort_values("PA")
by_shop.plot.barh(y="PA", figsize=(6, 7), legend=False, title="PA by shop")
by_shop.head(8)

### Nhóm lỗi

Mỗi câu chưa đúng thuộc một nhóm: không mẫu nào đúng (cần model / prompt tốt hơn), có mẫu đúng nhưng nhóm sai thắng phiếu, hoặc nhóm đúng
đứng đầu nhưng dưới ngưỡng nên bỏ trống (cần bỏ phiếu / ngưỡng tốt hơn).

In [ ]:
def kind(r):
    g = gold[r["id"]]
    s = scoring.score(r["final"], g)
    if s == 1: return "right"
    if not any(p and scoring.score(p, g) == 1 for p in (x["pred"] for x in r["samples"])): return "no right sample"
    if s == -1: return "wrong: a wrong group won"
    top_right = bool(r.get("vote")) and scoring.score(r["vote"], g) == 1
    return "abstain: right group on top, below threshold" if top_right else "abstain: right sample, another group on top"
pd.Series(Counter(kind(r) for r in trace.values())).sort_values(ascending=False).to_frame("questions")

### Ngưỡng bỏ phiếu và oracle@K

PA khi đổi ngưỡng đồng thuận (bỏ phiếu lại trên các mẫu đã lưu), và tỷ lệ câu có ít nhất một mẫu đúng (trần của việc chọn đáp án).

In [ ]:
def replay(r, tau):
    texts, preds = [s["text"] for s in r["samples"]], [s["pred"] for s in r["samples"]]
    pool = voting.debt_pool(texts, 8, bool(DEBT_WORDS.search(norm(r["question"]))))
    ans, conf = voting.vote([preds[j] for j in pool], len(pool))
    return voting.final_answer(ans, conf, tau, r["final"].get("reason") == "period outside the shop's data dates", True)
sweep = pd.Series({t: sum(scoring.score(replay(r, t), gold[i]) for i, r in trace.items()) / len(trace) for t in (0.25, 0.375, 0.5, 0.625, 0.75)}, name="PA")
oracle = sum(any(p and scoring.score(p, gold[i]) == 1 for p in (s["pred"] for s in r["samples"])) for i, r in trace.items()) / len(trace)
print(f"oracle (any sample right): {oracle:.3f}")
sweep.to_frame()

### Theo dạng câu hỏi

In [ ]:
from datamate import hints as H
shape = lambda q: ("debt" if DEBT_WORDS.search(norm(q)) else "profit" if H.PROFIT.search(norm(q)) else "trips" if H.TRIPS.search(norm(q))
                   else "lots" if H.LOT.search(norm(q)) else "other")
df = pd.DataFrame([{"shape": shape(r["question"]), "score": scoring.score(r["final"], gold[i])} for i, r in trace.items()])
df.groupby("shape")["score"].agg(n="size", PA="mean").sort_values("n", ascending=False).round(3)

### 2. So các bản nộp public (không nhãn)

Số câu trả lời, mức trùng đáp án giữa các file, và chỗ khác nhau theo shop giữa hai bản cuối.

In [ ]:
def key(a):
    t, x = a.get("type"), a.get("answer")
    if t == "abstain" or (x is None and t != "unanswerable"): return "ABSTAIN"
    return "UNANSWERABLE" if t == "unanswerable" else json.dumps([t, round(x, 2) if isinstance(x, float) else x], ensure_ascii=False, sort_keys=True)
pq = {q["id"]: q for q in read(env.data / "public_test/questions.jsonl")}
subs = {name: {r["id"]: key(r) for r in read(env.root / p / "submission.jsonl")} for name, p in SUBS.items() if (env.root / p / "submission.jsonl").exists()}
pd.DataFrame({n: {"answered": sum(v != "ABSTAIN" for v in s.values()), "unanswerable": sum(v == "UNANSWERABLE" for v in s.values())} for n, s in subs.items()}).T

In [ ]:
names = list(subs)
pd.DataFrame([[sum(subs[a][i] == subs[b][i] for i in pq) / len(pq) for b in names] for a in names], index=names, columns=names).style.format("{:.1%}")

In [ ]:
if len(names) >= 2:
    a, b = names[-2], names[-1]
    display(pd.DataFrame([{"shop": q["business_id"], "changed": subs[a][i] != subs[b][i], "newly answered": subs[a][i] == "ABSTAIN" != subs[b][i],
                           "newly abstained": subs[b][i] == "ABSTAIN" != subs[a][i]} for i, q in pq.items()]).groupby("shop").sum())

## 04 · Dữ liệu train LoRA

Từ các lượt chạy **trên train** (có đáp án) dựng các dòng SFT cho LoRA, theo công thức của LoRA v3:

1. **Lượt chạy có nhãn:** `02_inference` với `SPLIT = "train"`, `REUSE_LAYERS = False`, `SAVE_PROMPTS = True` (ví dụ `IDS = "splits/rft_train24.txt"`,
   2368 câu của 24 shop ngoài holdout). Mỗi câu có 8 mẫu thường và tới 4 mẫu thinking, kèm prompt đã gửi.
2. **Dòng SFT** (`training/build_sft.py`): mỗi câu giữ lời giải đúng trên route tốt nhất (macro công nợ → view chuẩn → bảng thô), câu khó
   (0-5/8 mẫu đúng) cho tới 2 dòng, câu dễ góp `EASY` dòng, mẫu thinking đúng thành dòng thinking. Lượt teacher (model lớn hơn chạy trên
   các câu 9B chưa giải được, chỉ trên train) đưa vào bằng `TEACHER_RUNS`.
3. **Ghi chú** (`training/add_notes.py`, cần GPU): dòng thường chưa có ghi chú được thêm vài dòng "Notes:" trước SQL, đúng định dạng lúc suy luận.
4. **Ngân hàng few-shot** (`training/build_fewshot_bank.py`), tùy chọn: mỗi câu một lời giải ngắn chỉ đọc view chuẩn.

Chỉ dùng dữ liệu train. Câu public / private không bao giờ vào dữ liệu train.

In [ ]:
install("llm")

In [ ]:
RUNS = ["runs/sql_v8_pool"]        # labeled train runs with --save-prompts (dirs holding trace.jsonl or shard*/trace.jsonl)
TEACHER_RUNS = []                  # optional teacher runs on train questions
OUT = "runs/sft/sql_rft_nb.jsonl"
EASY = 800                         # rows from easy questions (6-8 of 8 samples right)
MAX = 4000                         # cap on rows (debt and teacher rows are kept)
ADD_NOTES = True                   # needs a GPU (vLLM, base weights)
BUILD_BANK = False                 # also write a few-shot bank from RUNS

In [ ]:
def trace_dir(r):  # build_sft.py reads RUN/trace.jsonl: join the shards of a multi-GPU run first
    r = Path(r) if Path(r).is_absolute() else env.root / r
    if not (r / "trace.jsonl").exists() and list(r.glob("shard*/trace.jsonl")):
        with (r / "trace.jsonl").open("w", encoding="utf-8") as f:
            for p in sorted(r.glob("shard*/trace.jsonl")):
                f.write(p.read_text(encoding="utf-8"))
    return r
runs, teachers = [trace_dir(r) for r in RUNS], [trace_dir(r) for r in TEACHER_RUNS]
out = Path(OUT) if Path(OUT).is_absolute() else env.root / OUT
out.parent.mkdir(parents=True, exist_ok=True)
env.run([env.python, "training/build_sft.py", *runs, *(["--teacher", *teachers] if teachers else []), "--fewshot", "assets/fewshot_bank.jsonl",
         "--out", out, "--max", MAX, "--easy", EASY], log=out.with_suffix(".log"), show="available|rows by|Traceback|Error")

In [ ]:
sft = out
if ADD_NOTES:
    flags, _ = env.inference_flags()                     # checks the GPU, Ampere runs bf16 (no fp8)
    sft = out.with_name(out.stem + "_notes.jsonl")
    env.run([env.python, "training/add_notes.py", out, sft, "--model", env.base_model(), "--rulebook", env.data / "train/rulebook.md",
             "--quant", "fp8" if "--quant" in flags else "none"], log=sft.with_suffix(".log"), show="notes on|Traceback|Error")
print("SFT file for 05_train_lora:", sft)

### Dữ liệu SFT trông thế nào

Số dòng theo route, tỷ lệ dòng thinking, độ dài (prompt chiếm phần lớn mỗi dòng, loss chỉ tính trên phần trả lời).

In [ ]:
rows = read(sft)
df = pd.DataFrame([{"id": r["id"], "route": r.get("route"), "thinking": bool(r.get("thinking")), "prompt_chars": len(r["system"]) + len(r["user"]),
                    "reply_chars": len(r["reply"]), "notes": r["reply"].startswith("Notes:")} for r in rows])
print(len(df), "rows,", df["id"].nunique(), "questions")
display(df.groupby(["route", "thinking"]).size().unstack(fill_value=0))
display(df[["prompt_chars", "reply_chars"]].describe().round(0))
print(f"plain rows with notes: {df[~df.thinking].notes.mean():.1%}")

In [ ]:
r = rows[0]
print(r["user"][-600:], "\n---- reply ----\n", r["reply"][:800])

In [ ]:
if BUILD_BANK:
    env.run([env.python, "training/build_fewshot_bank.py", *runs, "--out", out.with_name("fewshot_bank.jsonl")], show="Traceback|Error|bank|rows")

## 05 · Train LoRA

QLoRA trên Qwen3.5-9B bằng Unsloth (`training/train_lora.py`): base 4-bit, LoRA r16 trên attention và MLP, loss chỉ trên phần trả lời.
LoRA v3 (bản đã nộp) là chạy này với dữ liệu của `04_training_data`, train tiếp từ v2: 2910 dòng, 364 bước, khoảng 8,2 giờ trên một GPU 24 GB.

**Độ dài dòng:** mỗi dòng dài từ 9k đến 19k token (trung vị khoảng 12,8k) vì prompt chứa rulebook và schema của shop. Loss chỉ tính trên phần trả lời
(khoảng 4% số token). `MAX_LEN = 16384` giữ khoảng 92% số dòng, `24576` giữ hết, dưới khoảng 9k thì không còn dòng nào.

**Phần cứng:**
- Công thức đầy đủ (`MAX_LEN = 32768`) đã chạy trên GPU bf16 24 GB (RTX PRO 4000 Blackwell, đỉnh 15,6 GiB, khoảng 80 giây mỗi bước 8 dòng).
- GPU 16 GB fp16 (T4, P100): chưa thử, có thể thử `MAX_LEN = 16384` với `LIMIT_ROWS` nhỏ để học, nhưng có thể hết bộ nhớ, và kernel fla có thể không chạy.
- Phiên Kaggle có giới hạn thời gian: checkpoint được lưu mỗi `SAVE_STEPS` bước vào `runs/` (Kaggle: `/kaggle/working`, giữ lại sau phiên),
  chạy lại notebook với `RESUME = True` (đính kèm output của phiên trước làm dataset rồi chép vào `OUT`) để train tiếp.

**Môi trường:** stack train (Unsloth, torch, peft, fla) khác môi trường suy luận. Máy local: `.venv-train`, Kaggle: notebook này tự cài
(dùng một phiên riêng, không chạy chung với 02 / 04).

In [ ]:
install("train")  # Kaggle: a separate session for this part (the training stack differs from vLLM)

In [ ]:
SFT = "runs/sft/sql_rft_nb_notes.jsonl"   # from 04_training_data (the v3 rows: data/sft/sql_rft_v3_notes.jsonl)
INIT_ADAPTER = "adapters/sql_rft_v3"      # continue from an adapter (same r and modules), or None to start fresh
OUT = "runs/lora/sql_rft_nb"
MAX_LEN = 32768                            # tokens per row, longer rows are dropped
LR = 5e-5                                  # v3: 5e-5 continuing from v2 (1e-4 from scratch)
EPOCHS = 1
GRAD_ACCUM = 8
LIMIT_ROWS = None                          # e.g. 64 for a short learning run
MAX_STEPS = -1                             # e.g. 3 for a speed / memory check
SAVE_STEPS = 50
RESUME = True                              # continue from the newest checkpoint in OUT
COMPLETION_LOGITS = True                   # LM head only on reply positions: same loss, ~12% faster

In [ ]:
py = str(env.root / ".venv-train/bin/python") if (env.root / ".venv-train/bin/python").exists() else env.python   # the training stack
out = Path(OUT) if Path(OUT).is_absolute() else env.root / OUT
sft = Path(SFT) if Path(SFT).is_absolute() else env.root / SFT
assert sft.exists(), f"{sft} not found: build it in 04_training_data"
print("training python:", py, "| precision:", env.training_precision(), "| rows:", sum(1 for _ in open(sft, encoding="utf-8")))

### Kernel fla

24 trên 32 lớp của Qwen3.5-9B là Gated DeltaNet. Có `fla-core` thì dùng kernel Triton (nhanh khoảng 90 lần mỗi lớp). Không có thì transformers lặng lẽ
dùng bản PyTorch tham chiếu. Ô này so kernel với bản tham chiếu trên dữ liệu ngẫu nhiên. Trượt thì chạy không có fla (đúng nhưng chậm).

In [ ]:
extra_env = {}
try:
    env.run([py, "training/check_fla.py", "--T", "2048"], show="fla check|T=|rel L2|Traceback|Error")
except RuntimeError as e:
    print("fla unusable here, training without it:", e)
    extra_env = {"PYTHONPATH": str(env.root / "training/no_fla")}

In [ ]:
cmd = [py, "training/train_lora.py", sft, "--out", out, "--model", env.base_model(), "--max-len", MAX_LEN, "--r", 16, "--alpha", 32, "--lr", LR,
       "--epochs", EPOCHS, "--grad-accum", GRAD_ACCUM, "--end", "<|im_end|>", "--no-thinking", "--max-steps", MAX_STEPS, "--save-steps", SAVE_STEPS]
cmd += (["--init-adapter", env.adapter(INIT_ADAPTER)] if INIT_ADAPTER else []) + (["--limit-rows", LIMIT_ROWS] if LIMIT_ROWS else [])
cmd += (["--resume"] if RESUME else []) + (["--completion-logits"] if COMPLETION_LOGITS else [])
env.run(cmd, log=out.with_name(out.name + ".log"), env=extra_env, show="examples|tokens:|precision|resuming|continuing|'loss'|steps in|Traceback|Error|OutOfMemory", every=120)

### Loss

Loss theo bước (đọc từ `trainer_state.json` của checkpoint mới nhất, hoặc từ log).

In [ ]:
import ast
log_lines = out.with_name(out.name + ".log").read_text(encoding="utf-8", errors="replace").replace("\r", "\n").splitlines()
losses = [float(ast.literal_eval(l.strip())["loss"]) for l in log_lines if l.strip().startswith("{'loss'")]
if losses:
    s = pd.Series(losses)
    ax = s.plot(alpha=0.3, figsize=(7, 3), title="training loss")
    s.rolling(max(1, len(s) // 20)).mean().plot(ax=ax)
print(len(losses), "logged steps; adapter:", out / "adapter", "exists" if (out / "adapter/adapter_model.safetensors").exists() else "MISSING")

### Dùng adapter vừa train

Trong `02_inference` đặt `LORA` bằng đường dẫn `OUT/adapter` (ví dụ `"runs/lora/sql_rft_nb/adapter"`), chạy trên holdout của train
(`SPLIT = "train"`, `IDS = "splits/holdout600.txt"`) hoặc public, rồi so trong `03_evaluation`. Holdout chỉ là smoke test: adapter đã học các
dòng của chính các câu đó (docs/leaderboard.md).